In [1]:
import pandas as pd
import json
import ast
#df = pd.read_json("inspect.json")

with open("inspect.json", "r") as f:
    res = json.load(f)

df = pd.json_normalize( res )
df.to_csv('inspect_clean.csv', index=False)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   record_id                       48 non-null     str   
 1   fields.Base Increment per Year  48 non-null     int64 
 2   fields.Base Salary              48 non-null     int64 
 3   fields.Department               48 non-null     object
 4   fields.Qualification            48 non-null     object
 5   fields.Qualified Position       48 non-null     object
 6   fields.base bonus.type          48 non-null     int64 
 7   fields.base bonus.value         48 non-null     object
 8   fields.base pay.type            48 non-null     int64 
 9   fields.base pay.value           48 non-null     object
 10  fields.lookup_id.type           48 non-null     int64 
 11  fields.lookup_id.value          48 non-null     object
dtypes: int64(5), object(6), str(1)
memory usage: 5.4+ KB


In [244]:
def general_standard_clean( df : pd.DataFrame ) -> pd.DataFrame:
    minus = df.select_dtypes(include= [ 'float', 'int', 'str'] ).columns.tolist()
    for c in [ c for c in df.columns.tolist() if c not in minus ]:
        df[c] = df[c].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    
        try:          
            df[c] = df[c].apply( lambda x: x[0]['text'] if x[0]['text'] else x )
        except:
            pass
        
    for c in [c for c in df.columns if 'date' in c.lower()]:
        df[c] = pd.to_datetime( 
                    df[c], unit='ms', utc=True 
                        ).dt.tz_convert('Asia/Kuala_Lumpur')

    df.columns = df.columns.str.replace( 'fields.', '' ).str.replace('.',  '_').str.lower()
    use_col = [  c for c in df.columns if 'type' not in c.lower() ]
    df = df[ use_col ]

    return df

In [245]:
test = general_standard_clean( df )
print(test)
test.to_csv('inspect_clean.csv', index=False)
#test = df.copy()
#test['base increment per year'] = test['base increment per year'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
#print(test['base increment per year'])

           record_id  base increment per year  base salary  \
0   reczz28PB2kCdOk3                      100         1700   
1   reczz28PB2kChzVy                      150         2200   
2   reczz28PB2kCjdmc                      175         3000   
3   reczz28PB2kCgVTZ                      200         4500   
4   reczz28PB2kChZRb                      200         6000   
5   reczz28PB6Sn71Bd                       50         1500   
6   reczz28PB6Sn56WV                      100         1700   
7   reczz28PB6Sn6S28                      150         2200   
8   reczz28PB6qIWRa6                      175         3000   
9   reczz28PB7i3nPff                      200         4500   
10  reczz28PB7i3nDu0                      200         6000   
11  reczz28PB7i3n4Wh                       50         1700   
12  reczz28PB7i3kbeX                      100         2200   
13  reczz28PB7v1nIqc                      150         3000   
14  reczz28PB7v1mJI3                      175         4500   
15  recz

In [ ]:
for x in [ 'qualified position', 'base bonus_value', 'base pay_value']:  
    test[x] = test[x].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    test[x] = test[x].apply(lambda x: x[0] if len(x) <= 1 else x)

    #if isinstance( text, list):
    #    print('yeah')
    #else:
    #    print('nope')


In [256]:
test = test.explode('qualified position')
test.to_csv('inspect_clean.csv', index=False)


In [254]:
example = "['Staff', 'Senior Staff']"
example = ast.literal_eval(example)
print(isinstance(example, list))

True


In [32]:
import pandas as pd
import ast
df = pd.read_csv( 'pull_raw.csv')

In [33]:
df['billable.UBUNTU.job_runs'] = \
df['billable.UBUNTU.job_runs'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df.explode('billable.UBUNTU.job_runs', ignore_index=True)

,run_duration_ms,billable.UBUNTU.total_ms,billable.UBUNTU.jobs,billable.UBUNTU.job_runs,id
0,80000,0,1,"{'job_id': 125078084915, 'duration_ms': 0}",41383144169
1,217000,0,1,"{'job_id': 100201593698, 'duration_ms': 0}",33615921523
2,198000,0,1,"{'job_id': 100200386500, 'duration_ms': 0}",33615538611
3,2378000,0,1,"{'job_id': 100187643441, 'duration_ms': 0}",33611548649
4,31000,0,1,"{'job_id': 100182801264, 'duration_ms': 0}",33610046400
5,127000,0,1,"{'job_id': 100180513085, 'duration_ms': 0}",33609322827
6,33000,0,1,"{'job_id': 100177380324, 'duration_ms': 0}",33608330104
7,86000,0,1,"{'job_id': 100176871638, 'duration_ms': 0}",33608169738
8,48000,0,1,"{'job_id': 100176132893, 'duration_ms': 0}",33607940232
9,147000,0,1,"{'job_id': 100174657636, 'duration_ms': 0}",33607469375


In [38]:
df['job_id'] = df['billable.UBUNTU.job_runs'].apply( 
    lambda x : x[0].get( 'job_id') if x[0] else None
) 

df['job_id_duration_ms'] = df['billable.UBUNTU.job_runs'].apply( 
    lambda x : x[0].get( 'duration_ms') if x[0] else None
)

In [39]:
df[['job_id', 'job_id_duration_ms']]

,job_id,job_id_duration_ms
0,125078084915,0
1,100201593698,0
2,100200386500,0
3,100187643441,0
4,100182801264,0
5,100180513085,0
6,100177380324,0
7,100176871638,0
8,100176132893,0
9,100174657636,0
